# 03 — Лаборатория: одиночные опционы и акция — строим, резюмируем, сравниваем

Вы построите все шесть конструкций через `strategies.*`, прочитаете риск каждой через
`analyzer.summarize`, нарисуете выплаты через `viz.plot_payoff` и поставите покрытый колл лицом к
лицу с голой акцией.

DEMO: спот **$100**, IV **0.25**, **45 DTE**. Середины рынка (mid) из цепочки модуля 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz
from optionslab.position import StockLeg

SPOT, VOL, t = 100.0, 0.25, 45/365

## Вспомогательная функция: аккуратный вывод сводки

`analyzer.summarize` возвращает словарь (чистая премия, точки безубыточности, макс. прибыль/убыток,
POP, ожидаемое движение, греки, DTE). Мы форматируем ключевые поля.

In [ ]:
def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  (+дебет/-кредит)")
    print(f"  breakevens  {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}   max_loss {s['max_loss']:.0f}")
    print(f"  POP {s['probability_of_profit']:.2f}   delta {s['greeks'].delta:+.1f}")

## 1. Длинный колл и длинный пут (дебет, длинная вега, короткая тета)

In [ ]:
lc = strategies.long_call((100, 3.91), expiry=t)
lp = strategies.long_put((100, 3.42), expiry=t)
show(lc); print(); show(lp)

Обратите внимание на точку безубыточности длинного колла — **103.91** (страйк + премия) — и на
ограниченный максимальный убыток, равный дебету. У обоих POP < 0.5: акция должна *двигаться*, причём
достаточно, чтобы отбить премию.

## 2. Покрытый колл и обеспеченный деньгами пут (кредит, короткая вега, длинная тета)

In [ ]:
cc  = strategies.covered_call(100, (105, 1.85), expiry=t)
csp = strategies.cash_secured_put((95, 1.58), expiry=t)
show(cc); print(); show(csp)

Покрытый колл ограничивает максимальную прибыль на **685** (5 пунктов роста + кредит 1.85), а CSP
оставляет себе кредит **158**, если DEMO удержится выше 95. У обоих положительная тета и высокий
POP — профиль продавца.

## 3. Защитный пут и коллар (захеджированная акция)

In [ ]:
pp = strategies.protective_put(100, (95, 1.58), expiry=t)
col = strategies.collar(100, (95, 1.58), (105, 1.85), expiry=t)
show(pp); print(); show(col)

Защитный пут ставит пол под максимальным убытком на **-658**; коллар запирает исход примерно в
**-473 / +527** за небольшой чистый кредит — проданный колл оплачивает купленный пут.

## 4. Диаграмма выплат: длинный колл

`viz.plot_payoff` рисует линию выплат на экспирации; параметр `vol` добавляет кривую переоценки по
модели «на сейчас», а `spot` отмечает текущую цену.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_payoff(lc, spot=SPOT, vol=VOL, ax=ax)
ax.set_title('Длинный 100-й колл — ограниченный убыток, открытый потенциал вверх')
plt.show()

## 5. Покрытый колл против голой акции

Соберите голую длинную акцию через `strategies.custom` + `StockLeg`, затем наложите обе кривые
выплат на экспирации через `viz.plot_compare`. Посмотрите, где именно кредит помогает, а потолок
мешает.

In [ ]:
stock = strategies.custom(StockLeg(100, SPOT), label='Длинные 100 акций')
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_compare([stock, cc], ax=ax)
ax.set_title('Покрытый колл против голой акции (P&L на экспирации)')
plt.show()

Ниже ~105 покрытый колл идёт **выше** акции на величину премиальной подушки; выше 105 акция
продолжает расти, а покрытый колл выходит на плато на своём потолке. Вы продали хвост роста за
гарантированный кредит.

## 6. Где они пересекаются? (посчитаем размен)

Через `payoff.pnl_curve` посчитайте оба P&L на сетке и найдите численно точку пересечения и подушку
на снижении.

In [ ]:
grid = np.linspace(85, 120, 71)
pnl_stock = payoff.pnl_curve(stock, grid)
pnl_cc    = payoff.pnl_curve(cc, grid)
cushion = grid[np.argmin(np.abs(pnl_cc))]      # безубыточность покрытого колла ~98.15
cross   = grid[np.argmin(np.abs(pnl_cc - pnl_stock))]
print(f'безубыточность покрытого колла около спота {cushion:.1f}')
print(f'покрытый колл и акция дают равный P&L около спота {cross:.1f} (короткий страйк)')

## Эксперименты

1. В разделе 2 продайте для покрытого колла **107.5** колл вместо 105 (mid ~1.19). Роста остаётся
   больше, премии — меньше: как изменятся max_profit и подушка?
2. В разделе 1 купите **105** колл (~1.85) вместо ATM. Сравните точку безубыточности и POP — чем
   более дешёвый и дальний OTM-страйк обходится вам по вероятности?
3. В разделе 3 расширьте коллар до **90-го пута / 110-го колла**. Останется ли он кредитным? Как
   изменится коридор максимального убытка и максимальной прибыли?
4. Прогоните `analyzer.summarize` по покрытому коллу при `vol=0.40` вместо 0.25. Какие поля
   сдвинутся и почему продавец покрытых коллов *предпочитает* более высокую IV на входе?
5. Соберите обеспеченный деньгами пут на страйке **90** (~0.62). Сравните его POP и максимальный
   убыток с версией на 95 — классический размен «дальше OTM = выше POP, меньше кредит».